# Generate Test Data
Use ffmpeg to extract segments from the src videos so I can have smaller test segments

In [2]:
import subprocess

## 1. GRAPPLING HOOK!

In [15]:
# Get the video clip
subprocess.run(
    "echo y | ../tools/ffmpeg  -ss 00:19:40.000 -i Assets/src.mp4 -t 5 -map 0 -c copy Assets/1.mp4",
    shell=True,
    capture_output=True
)

CompletedProcess(args='echo y | ../tools/ffmpeg  -ss 00:19:40.000 -i Assets/src.mp4 -t 5 -map 0 -c copy Assets/1.mp4', returncode=0, stdout=b'', stderr=b"ffmpeg version 6.1-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2023 the FFmpeg developers\n  built with gcc 8 (Debian 8.3.0-6)\n  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libfribidi --enable-libass --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg\n  libavutil

## 2. There's a goat on my bed

In [16]:
# Get the video clip
subprocess.run(
    "echo y | ../tools/ffmpeg  -ss 00:01:45.000 -i Assets/src.mp4 -t 7 -map 0 -c copy Assets/2.mp4",
    shell=True,
    capture_output=True
)

CompletedProcess(args='echo y | ../tools/ffmpeg  -ss 00:01:45.000 -i Assets/src.mp4 -t 7 -map 0 -c copy Assets/2.mp4', returncode=0, stdout=b'', stderr=b"ffmpeg version 6.1-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2023 the FFmpeg developers\n  built with gcc 8 (Debian 8.3.0-6)\n  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libfribidi --enable-libass --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg\n  libavutil

## 3. There's something I should tell you...

In [17]:
# Get the video clip
subprocess.run(
    "echo y | ../tools/ffmpeg  -ss 00:12:26.000 -i Assets/src.mp4 -t 60 -map 0 -c copy Assets/3.mp4",
    shell=True,
    capture_output=True
)

CompletedProcess(args='echo y | ../tools/ffmpeg  -ss 00:12:26.000 -i Assets/src.mp4 -t 60 -map 0 -c copy Assets/3.mp4', returncode=0, stdout=b'', stderr=b"ffmpeg version 6.1-static https://johnvansickle.com/ffmpeg/  Copyright (c) 2000-2023 the FFmpeg developers\n  built with gcc 8 (Debian 8.3.0-6)\n  configuration: --enable-gpl --enable-version3 --enable-static --disable-debug --disable-ffplay --disable-indev=sndio --disable-outdev=sndio --cc=gcc --enable-fontconfig --enable-frei0r --enable-gnutls --enable-gmp --enable-libgme --enable-gray --enable-libfribidi --enable-libass --enable-libfreetype --enable-libmp3lame --enable-libopencore-amrnb --enable-libopencore-amrwb --enable-libopenjpeg --enable-librubberband --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libvorbis --enable-libopus --enable-libtheora --enable-libvidstab --enable-libvo-amrwbenc --enable-libvpx --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg\n  libavuti

# Get duration

In [ ]:
import json
import subprocess
result = subprocess.run(
    "../tools/ffprobe  -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 Assets/src.mp4",
    shell=True,
    capture_output=True
)
runtime_secs = json.loads(result.stdout)
runtime_secs


1262.817234

In [5]:
# According to VLC it's 21:02 runtime
# Which is obviously
21*60+2

1262

# Extract Audio

In [19]:
# Extract Audio
import pathlib
for file in pathlib.Path("Assets").glob("*.mp4"):
    if "src.mp4" == file.name:
        continue
    print( file.name )
    subprocess.run(
        f"echo y | ../tools/ffmpeg  -i Assets/{file.name} Assets/{file.stem}.wav",
        shell=True,
        capture_output=True
    )


3.mp4
2.mp4
1.mp4


# Generate Subtitles

In [1]:
def format_timestamp(sec : float):
    hh = int(sec // 3600)
    mm = int((sec % 3600) // 60)
    ss = int(sec % 60)
    mil = int((sec - int(sec)) * 1000)
    return f"{hh:02}:{mm:02}:{ss:02}.{mil:03}"


In [2]:
import pathlib
from faster_whisper import WhisperModel
model = WhisperModel(
    'small.en', device="cpu", compute_type="int8", download_root="tools"
)
for audio_path in pathlib.Path("Assets").glob("*.wav"):
    srt_file = audio_path.with_suffix(".srt")
    segments, _ = model.transcribe(audio_path, beam_size=5)

    with srt_file.open(mode="w") as f:
        for i, segment in enumerate(segments):
            # Skip lines that are descriptions (e.g. '[music]', '[applause]')
            if segment.text.endswith("]"):
                continue
            f.write(
                f"{format_timestamp(segment.start)} --> {format_timestamp(segment.end)}\n{segment.text.strip()}\n\n"
            )
        pass

# Create videos with subtitles

In [ ]:
import pathlib
import subprocess

for video_file in pathlib.Path("Assets").glob("*.mp4"):
    srt_file = video_file.with_suffix(".srt")
    if not srt_file.exists():
        continue
    output_video = pathlib.Path(f"{video_file.stem}_with_subtitles.mp4")
    subprocess.run(
        f"echo y | ffmpeg -i {video_file} -f srt -i {srt_file} -c copy -c:s mov_text {output_video}",
        shell=True
    )

ffmpeg version 7.1.1-1ubuntu4.2 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 15 (Ubuntu 15.2.0-4ubuntu4)
  configuration: --prefix=/usr --extra-version=1ubuntu4.2 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-libmfx --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --